# 사람 탐지 모델 학습

공용 GPU 서버에서 사람 탐지(Person Detection) 모델을 fine-tuning하는 노트북이다.
범위와 준비물은 [`../README.md`](../README.md)를 먼저 읽는다.

**셀은 위에서 아래로 순서대로, 한 번씩 실행한다.** 어떤 셀이 실패하면 그 셀만
다시 실행하지 말고 바로 위 셀부터 원인을 확인한다 — 뒤 셀이 앞 셀의 변수를 그대로 쓴다.

시작하기 전에 같은 디렉터리에 `.env`가 있는지 확인한다. 없다면 터미널에서
`cp .env.example ../.env` 대신 `deeplearning/training/`에서
`cp .env.example .env`를 실행하고 값을 채운 뒤 이 노트북을 다시 연다.

## 1. 환경 점검

Python·PyTorch·CUDA가 기대한 대로 잡혀 있는지 확인한다. `CUDA 사용 가능: False`가
나오면 `DEVICE=cuda`로 학습해도 실제로는 CPU로 돈다 — 서버의 CUDA/드라이버 설치를
먼저 확인한다.

In [ ]:
import platform
import sys

print(f"Python: {sys.version}")
print(f"플랫폼: {platform.platform()}")

try:
    import torch
except ImportError as exc:
    raise SystemExit(
        "torch가 설치되어 있지 않다. 이 디렉터리에서 "
        "`pip install -r requirements.txt`를 먼저 실행한다."
    ) from exc

print(f"torch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 2. GPU 사용 현황 확인

**공용 GPU를 여러 명이 나눠 쓴다.** 이미 메모리를 크게 쓰는 프로세스가 있으면
학습을 시작하기 전에 팀 채널에서 먼저 확인한다.

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ["nvidia-smi"], capture_output=True, text=True, check=True
    )
    print(result.stdout)
except FileNotFoundError:
    print("nvidia-smi를 찾을 수 없다. GPU 드라이버가 설치된 서버에서 실행 중인지 확인한다.")
except subprocess.CalledProcessError as exc:
    print(f"nvidia-smi 실행 실패: {exc.stderr}")

## 3. 설정 로드

`../.env`의 값을 읽는다. 필수 값이 없으면 여기서 바로 멈춘다 — 학습을 다 돌리고
나서야 경로가 잘못됐다는 것을 알게 되는 상황을 피하기 위해서다.

In [ ]:
from pathlib import Path

from dotenv import load_dotenv
import os

TRAINING_DIR = Path.cwd().resolve().parent  # notebooks/ 의 부모 = training/
load_dotenv(TRAINING_DIR / ".env")


def _require(name: str) -> str:
    value = os.environ.get(name, "").strip()
    if not value:
        raise SystemExit(
            f"필수 환경변수 {name}이 비어 있다. "
            f"{TRAINING_DIR / '.env'}를 채운 뒤 커널을 재시작한다."
        )
    return value


DATASET_DIR = Path(_require("DATASET_DIR"))
DEVICE = os.environ.get("DEVICE", "cuda")
BASE_WEIGHTS_PATH = os.environ.get("BASE_WEIGHTS_PATH", "yolov8n.pt")
EPOCHS = int(os.environ.get("EPOCHS", "100"))
IMAGE_SIZE = int(os.environ.get("IMAGE_SIZE", "640"))
BATCH_SIZE = int(os.environ.get("BATCH_SIZE", "16"))
_output_dir_value = os.environ.get("OUTPUT_DIR", "").strip()
OUTPUT_DIR = Path(_output_dir_value) if _output_dir_value else TRAINING_DIR / "runs"

print(f"DATASET_DIR: {DATASET_DIR}")
print(f"DEVICE: {DEVICE}")
print(f"BASE_WEIGHTS_PATH: {BASE_WEIGHTS_PATH}")
print(f"EPOCHS: {EPOCHS}, IMAGE_SIZE: {IMAGE_SIZE}, BATCH_SIZE: {BATCH_SIZE}")
print(f"OUTPUT_DIR: {OUTPUT_DIR}")

## 4. 데이터셋 확인

`DATASET_DIR`에 YOLO 포맷 데이터셋(`data.yaml`, `images/`, `labels/`)이 있다고
가정한다. 클래스 수와 이름, 이미지 장수를 학습 시작 전에 눈으로 확인한다.

In [ ]:
import yaml

data_yaml_path = DATASET_DIR / "data.yaml"
if not data_yaml_path.exists():
    raise SystemExit(
        f"{data_yaml_path}가 없다. DATASET_DIR이 YOLO 포맷 데이터셋 루트를 "
        "가리키는지 확인한다."
    )

with data_yaml_path.open(encoding="utf-8") as f:
    dataset_config = yaml.safe_load(f)

print(f"클래스 수: {dataset_config.get('nc')}")
print(f"클래스 이름: {dataset_config.get('names')}")

for split in ("train", "val"):
    split_rel = dataset_config.get(split)
    if not split_rel:
        print(f"[경고] data.yaml에 {split} 경로가 없다.")
        continue
    split_dir = (DATASET_DIR / split_rel).resolve()
    count = (
        len(list(split_dir.glob("*.*"))) if split_dir.exists() else 0
    )
    print(f"{split}: {split_dir} — 파일 {count}개")

## 5. 샘플 이미지 확인 (선택)

라벨이 이미지와 제대로 맞는지 한 장만 눈으로 확인한다. `train` 스플릿의 첫 이미지를
그린다. 그림이 이상하면(박스 위치가 어긋나 있으면) 학습을 시작하기 전에
데이터셋 생성 과정을 먼저 점검한다.

In [ ]:
import matplotlib.patches as patches
import matplotlib.pyplot as plt
from PIL import Image

train_dir = (DATASET_DIR / dataset_config["train"]).resolve()
sample_images = sorted(train_dir.glob("*.jpg")) or sorted(train_dir.glob("*.png"))

if not sample_images:
    print(f"{train_dir}에서 이미지를 찾지 못했다. 확장자나 경로를 확인한다.")
else:
    sample_image_path = sample_images[0]
    label_path = (
        DATASET_DIR / "labels" / dataset_config["train"].split("/")[-1]
        / f"{sample_image_path.stem}.txt"
    )

    image = Image.open(sample_image_path)
    fig, ax = plt.subplots(1, figsize=(8, 8))
    ax.imshow(image)
    width, height = image.size

    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            cls_id, cx, cy, w, h = (float(v) for v in line.split())
            box_w, box_h = w * width, h * height
            x = cx * width - box_w / 2
            y = cy * height - box_h / 2
            rect = patches.Rectangle(
                (x, y), box_w, box_h, linewidth=2, edgecolor="lime", facecolor="none"
            )
            ax.add_patch(rect)
    else:
        print(f"[경고] 라벨 파일이 없다: {label_path}")

    ax.set_title(sample_image_path.name)
    plt.show()

## 6. 모델 로드

`BASE_WEIGHTS_PATH`가 로컬에 없으면 ultralytics가 자동으로 내려받는다.
사람 탐지 모델 버전은 아직 `결정 필요`([`deeplearning/README.md`](../../README.md#모델-선정))이므로
`.env`의 `BASE_WEIGHTS_PATH`만 바꾸면 다른 후보(`yolo11n.pt` 등)로도 그대로 실행된다.

In [ ]:
from ultralytics import YOLO

model = YOLO(BASE_WEIGHTS_PATH)
print(f"기준 가중치 로드 완료: {BASE_WEIGHTS_PATH}")

## 7. 학습 실행

결과는 `OUTPUT_DIR/person_detection/`에 쌓인다. 시간이 오래 걸릴 수 있다 —
GPU와 데이터셋 크기에 따라 다르므로 셀 실행 전에 대략적인 소요 시간을 팀원과
공유해 둔다.

In [ ]:
results = model.train(
    data=str(data_yaml_path),
    epochs=EPOCHS,
    imgsz=IMAGE_SIZE,
    batch=BATCH_SIZE,
    device=DEVICE,
    project=str(OUTPUT_DIR),
    name="person_detection",
    exist_ok=True,
)

## 8. 학습 결과 확인

검증 지표와 가장 좋은 가중치의 경로를 확인한다. `best.pt`가 다음 단계(공유,
`worker/inference`로의 전달)에서 쓸 파일이다.

In [ ]:
metrics = model.val()
print(metrics)

run_dir = Path(model.trainer.save_dir)
best_weights_path = run_dir / "weights" / "best.pt"
print(f"학습 결과 디렉터리: {run_dir}")
print(f"최고 성능 가중치: {best_weights_path}")
print(f"손실 곡선 등 그래프: {run_dir / 'results.png'}")

## 9. 디스크 사용량 확인과 정리

**공용 서버의 가용 용량이 약 17~20 GB뿐이다**
([결정 0011](../../../docs/architecture/decisions.md#0011--영상-원본을-저장하지-않고-스냅샷만-남긴다)).
가중치를 다른 곳으로 옮겼다면 `DATASET_DIR`과 `OUTPUT_DIR`을 지워 다음 사람이 쓸
공간을 남긴다. **아래 셀은 사용량만 보여주고 아무것도 지우지 않는다** — 삭제는
직접 확인하고 수동으로 한다.

In [ ]:
import shutil


def _dir_size_gb(path: Path) -> float:
    if not path.exists():
        return 0.0
    total_bytes = sum(f.stat().st_size for f in path.rglob("*") if f.is_file())
    return total_bytes / (1024 ** 3)


usage = shutil.disk_usage(TRAINING_DIR)
print(f"서버 디스크 — 전체: {usage.total / 1024**3:.1f} GB, "
      f"사용 중: {usage.used / 1024**3:.1f} GB, 여유: {usage.free / 1024**3:.1f} GB")
print(f"DATASET_DIR 크기: {_dir_size_gb(DATASET_DIR):.2f} GB")
print(f"OUTPUT_DIR 크기: {_dir_size_gb(OUTPUT_DIR):.2f} GB")

if usage.free / 1024 ** 3 < 5:
    print("[경고] 여유 공간이 5 GB 미만이다. 정리 전까지 다른 학습을 새로 시작하지 않는다.")

## 마무리

- `best.pt`를 필요한 사람과 팀 채널로 공유한다. **저장소에 커밋하지 않는다** —
  `.gitignore`가 `*.pt`와 이 디렉터리 아래 `runs/`, `data/`를 막는다.
  운영 추론 환경(`worker/inference`)까지 전달하는 공식 경로는 아직
  `결정 필요`다([`../README.md`](../README.md#남은-일)).
- 학습이 끝나고 결과를 다 옮겼다면 `DATASET_DIR`과 `OUTPUT_DIR`을 정리해 다음
  사람이 쓸 디스크 공간을 남긴다.
- 이 노트북으로 만든 가중치는 사람 탐지 모델까지다. 얼굴 탐지·인식 학습은
  아직 이 저장소의 범위 밖이다.